In [1]:
import pandas as pd
import numpy as np
import os
import gc
import transformers
import torch
import logging
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
from tqdm.notebook import tqdm

logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
tqdm.pandas()

In [2]:
CATEGORIES = ["Мир", "Россия", "Экономика", "Наука и техника", "Спорт", "Культура"]
CATEGORIES_STR = ", ".join(CATEGORIES)

N_SHOTS    = 2    # примеров на класс
EXAMPLE_LEN = 300  # символов в каждом примере

DATA_PATH   = Path("news_data/cleaned_news_for_model.parquet")
SAMPLE_PATH = Path("news_data/llm_sample_1200.parquet")
REPORT_PATH = Path("reports/llm_fewshot_results.csv")
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [3]:
# Загружаем ту же выборку что использовалась в zero-shot
df_sample = pd.read_parquet(SAMPLE_PATH)
print(f"Тестовая выборка: {len(df_sample)} строк")

# Для примеров берём всё что НЕ попало в выборку
df = pd.read_parquet(DATA_PATH)
df_model = df[["title", "text", "category_raw"]].dropna().copy()
df_model["llm_text"] = df_model["text"].astype(str)
df_model = df_model[df_model["llm_text"].str.len() > 0]

df_pool = df_model[~df_model.index.isin(df_sample.index)]
print(f"Пул для примеров: {len(df_pool)} строк")

Тестовая выборка: 1200 строк
Пул для примеров: 145419 строк


In [4]:
few_shot_examples = []
for cat in CATEGORIES:
    cat_examples = (
        df_pool[df_pool["category_raw"] == cat]
        .sample(n=N_SHOTS, random_state=42)
    )
    for _, row in cat_examples.iterrows():
        few_shot_examples.append({
            "category": cat,
            "text": row["llm_text"][:EXAMPLE_LEN],
        })

# Проверяем что примеры не пересекаются с тестовой выборкой по индексу
example_idx = {ex["text"] for ex in few_shot_examples}
leaks = df_sample["llm_text"].apply(lambda t: t[:EXAMPLE_LEN] in example_idx).sum()
assert leaks == 0, f"Утечка! {leaks} примеров из промпта есть в выборке"

print(f"Few-shot примеров: {len(few_shot_examples)} ({N_SHOTS} на класс × {len(CATEGORIES)} классов)")
print(f"Каждый пример — первые {EXAMPLE_LEN} символов")

# Показываем что получилось
for ex in few_shot_examples:
    print(f"\n[{ex['category']}] {ex['text'][:80]}...")

Few-shot примеров: 12 (2 на класс × 6 классов)
Каждый пример — первые 300 символов

[Мир] Politico: ЕС разрабатывает план по частичному членству Украины в 2027 году
Семен...

[Мир] Sky News: Лидеры ЕС могут поехать в США, чтобы повлиять на Трампа по Украине
Вик...

[Россия] Сенатор Косачев: Превращение БРИКС или ШОС в военный блок бесперспективно
Идеи п...

[Россия] Врачи НИИ Склифосовского провели уникальную трансплантацию кисти от донора
Росси...

[Экономика] Тайфун образовался к югу от Японии, он может затронуть Курильские острова
Тайфун...

[Экономика] Депутат Госдумы Нилов: Цены на цветы следует ограничить законодательно
Фото: Ser...

[Наука и техника] Daily Mail: Распятие Иисуса Христа могло произойти 3 апреля 33 года нашей эры
Ек...

[Наука и техника] eBioMedicine: У пациентов с длительным COVID повышается уровень тау-белка
Екатер...

[Спорт] Вратарь сборной России по водному поло Федотов: Были готовы играть с Украиной
Фо...

[Спорт] Лыжник Коростелев выполнил олимпийский нормат

In [5]:
SYSTEM_PROMPT_FS = (
    f"Ты классификатор новостных статей.\n"
    f"Определи категорию текста и ответь ТОЛЬКО одним из вариантов: {CATEGORIES_STR}.\n"
    f"Никаких пояснений — только одно слово или фраза из списка."
)

def build_few_shot_messages(text):
    """Собирает messages: system + примеры как диалог + целевой текст."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT_FS}]

    # Примеры — диалог где модель уже "отвечала" правильно
    for ex in few_shot_examples:
        messages.append({"role": "user",      "content": ex["text"]})
        messages.append({"role": "assistant", "content": ex["category"]})

    # Целевой текст — последний вопрос без ответа
    messages.append({"role": "user", "content": text})
    return messages

def classify_few_shot(text, pipeline, max_new_tokens=20):
    messages = build_few_shot_messages(text)
    output = pipeline(messages, max_new_tokens=max_new_tokens, do_sample=False)
    response = output[0]["generated_text"][-1]["content"].strip()

    for cat in CATEGORIES:
        if cat.lower() in response.lower():
            return cat

    print(f"[Неизвестно] Ответ модели: '{response}'")
    return "Неизвестно"

def free_memory(obj):
    del obj
    gc.collect()
    torch.mps.empty_cache()
    print("Память освобождена")

def compute_metrics(y_true, y_pred, model_name):
    mask = y_pred != "Неизвестно"
    accuracy    = accuracy_score(y_true[mask], y_pred[mask])
    macro_f1    = f1_score(y_true[mask], y_pred[mask], average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_true[mask], y_pred[mask], average="weighted", zero_division=0)
    unknown_rate = (~mask).mean()

    print(f"\n=== {model_name} ===")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print(f"Неизвестно:  {unknown_rate:.1%} ответов не распознано")
    print(classification_report(y_true[mask], y_pred[mask], zero_division=0))

    return accuracy, macro_f1, weighted_f1, unknown_rate

In [15]:
# Посмотреть как выглядит промпт перед запуском
test_messages = build_few_shot_messages("Тестовый текст новости")
for msg in test_messages:
    role = msg["role"].upper()
    preview = msg["content"][:100].replace("\n", " ")
    print(f"[{role}] {preview}...")
    print()

[SYSTEM] Ты классификатор новостных статей. Определи категорию текста и ответь ТОЛЬКО одним из вариантов: Мир...

[USER] Politico: ЕС разрабатывает план по частичному членству Украины в 2027 году Семен Александров (старши...

[ASSISTANT] Мир...

[USER] Sky News: Лидеры ЕС могут поехать в США, чтобы повлиять на Трампа по Украине Виктория Кондратьева (Р...

[ASSISTANT] Мир...

[USER] Сенатор Косачев: Превращение БРИКС или ШОС в военный блок бесперспективно Идеи превратить БРИКС или ...

[ASSISTANT] Россия...

[USER] Врачи НИИ Склифосовского провели уникальную трансплантацию кисти от донора Российские врачи НИИ скор...

[ASSISTANT] Россия...

[USER] Тайфун образовался к югу от Японии, он может затронуть Курильские острова Тайфун «Нари», пятый в это...

[ASSISTANT] Экономика...

[USER] Депутат Госдумы Нилов: Цены на цветы следует ограничить законодательно Фото: Sergey Elagin / Busines...

[ASSISTANT] Экономика...

[USER] Daily Mail: Распятие Иисуса Христа могло произойти 3 апреля 33 года н

In [7]:
pipeline_gemma = transformers.pipeline(
    "text-generation",
    model="google/gemma-4-E4B-it",
    dtype="auto",
    device_map="auto",
)
print("Gemma загружена")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Gemma загружена


In [8]:
print(f"Few-shot классификация {len(df_sample)} текстов через Gemma...")
df_sample["pred_gemma_fs"] = df_sample["llm_text"].progress_apply(
    lambda text: classify_few_shot(text, pipeline_gemma)
)
df_sample["pred_gemma_fs"].value_counts()

Few-shot классификация 1200 текстов через Gemma...


  0%|          | 0/1200 [00:00<?, ?it/s]

pred_gemma_fs
Мир                480
Россия             265
Экономика          187
Наука и техника    117
Спорт               82
Культура            69
Name: count, dtype: int64

In [9]:
free_memory(pipeline_gemma)

Память освобождена


In [10]:
acc_g, mf1_g, wf1_g, unk_g = compute_metrics(
    df_sample["category_raw"], df_sample["pred_gemma_fs"], "Gemma-4-E4B-it few-shot"
)


=== Gemma-4-E4B-it few-shot ===
Accuracy:    0.8150
Macro F1:    0.8253
Weighted F1: 0.8121
Неизвестно:  0.0% ответов не распознано
                 precision    recall  f1-score   support

       Культура       0.83      0.90      0.86        63
            Мир       0.75      0.97      0.85       371
Наука и техника       0.62      0.87      0.72        83
         Россия       0.89      0.70      0.78       339
          Спорт       0.96      0.98      0.97        81
      Экономика       0.93      0.66      0.77       263

       accuracy                           0.81      1200
      macro avg       0.83      0.85      0.83      1200
   weighted avg       0.84      0.81      0.81      1200



In [11]:
results = pd.DataFrame([
    {
        "experiment": "13_gemma4_e4b_fewshot",
        "model": "google/gemma-4-E4B-it",
        "input": "text", "classifier": f"few-shot LLM ({N_SHOTS}/class)",
        "sample_size": len(df_sample),
        "accuracy": acc_g, "macro_f1": mf1_g,
        "weighted_f1": wf1_g, "unknown_rate": unk_g,
    },
])

results.to_csv(
    REPORT_PATH,
    mode="a",
    header=not REPORT_PATH.exists(),
    index=False,
)
results

,experiment,model,input,classifier,sample_size,accuracy,macro_f1,weighted_f1,unknown_rate
0,13_gemma4_e4b_fewshot,google/gemma-4-E4B-it,text,few-shot LLM (2/class),1200,0.815,0.825287,0.812148,0.0


In [12]:
model_col = "pred_gemma_fs"  

errors = df_sample[df_sample["category_raw"] != df_sample[model_col]][[
    "llm_text", "category_raw", model_col
]].rename(columns={"llm_text": "text", "category_raw": "true", model_col: "pred"})

print(f"Ошибок: {len(errors)} из {len(df_sample)}")
errors.head(20)

Ошибок: 222 из 1200


,text,true,pred
107059,Фитнес-тренер упомянула еду после 18:00 и вред...,Спорт,Наука и техника
26262,На Западном берегу Иордана избили палестинског...,Культура,Мир
64495,РИА: Конфликт Таиланда и Камбоджи ударит по по...,Экономика,Мир
140930,Доцент Балынин: Шестидневных рабочих недель в ...,Россия,Экономика
50050,Средства от продаж в День мороженого в ГУМе пе...,Россия,Наука и техника
103749,Жительница Москвы согласилась на новые ключи о...,Экономика,Мир
56468,Shot: Четыре человека не выжили при падении са...,Россия,Наука и техника
75911,Машков: За два года «Театральный бульвар» стре...,Россия,Культура
38667,Telegram-канал «Управление Z»: ВСУ атаковали Т...,Россия,Мир
112850,Группировка «Запад» ВС РФ: Атаки ВСУ в Купянск...,Россия,Мир


In [14]:
import json

def build_few_shot_messages(text, cache_file="news_data/prompts/few_shots_2_per_class_300_symb.json"):
    """
    Собирает messages: system + примеры как диалог + целевой текст.
    
    Кеш хранит только базовую часть (system + few-shot примеры).
    Целевой text всегда добавляется в конец динамически.
    """
    if cache_file is not None:
        cache_path = Path(cache_file)

        if cache_path.exists():
            with cache_path.open("r", encoding="utf-8") as f:
                base_messages = json.load(f)
            return base_messages + [{"role": "user", "content": text}]

    # Строим базовую часть без целевого текста
    base_messages = [{"role": "system", "content": SYSTEM_PROMPT_FS}]
    for ex in few_shot_examples:
        base_messages.append({"role": "user",      "content": ex["text"]})
        base_messages.append({"role": "assistant", "content": ex["category"]})

    # Сохраняем только базовую часть
    if cache_file is not None:
        cache_path = Path(cache_file)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with cache_path.open("w", encoding="utf-8") as f:
            json.dump(base_messages, f, ensure_ascii=False, indent=2)

    return base_messages + [{"role": "user", "content": text}]